**Install Hugging Face libraries**

In [ ]:
!pip install -U transformers

**Importing Libraries**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer

**Importing Dataset**

In [ ]:
df = pd.read_csv("/content/df_file.csv")
print(df.head())
print(df.columns)

                                                Text  Label
0  Budget to set scene for election\n \n Gordon B...      0
1  Army chiefs in regiments decision\n \n Militar...      0
2  Howard denies split over ID cards\n \n Michael...      0
3  Observers to monitor UK election\n \n Minister...      0
4  Kilroy names election seat target\n \n Ex-chat...      0
Index(['Text', 'Label'], dtype='object')


In [ ]:
df.dtypes

,0
Text,object
Label,int64


**Train test split**

In [ ]:
train_df, test_df = train_test_split(
    df[["Text", "Label"]],
    test_size=0.2,
    random_state=42,
    stratify=df["Label"]
)

len(train_df), len(test_df)

(1780, 445)

**Convert to Hugging Face Dataset format**

In [ ]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

train_ds, test_ds

(Dataset({
     features: ['Text', 'Label'],
     num_rows: 1780
 }),
 Dataset({
     features: ['Text', 'Label'],
     num_rows: 445
 }))

**Loading the Hugging Face model & tokenizer**

In [ ]:
num_labels = df["Label"].nunique()
print("Number of labels:", num_labels)

model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Number of labels: 5


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**Tokenize the text**

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["Text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

train_tokenized = train_ds.map(tokenize_function, batched=True)
test_tokenized = test_ds.map(tokenize_function, batched=True)

train_tokenized = train_tokenized.rename_column("Label", "labels")
test_tokenized = test_tokenized.rename_column("Label", "labels")

train_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/1780 [00:00<?, ? examples/s]

Map:   0%|          | 0/445 [00:00<?, ? examples/s]

**Set up Trainer**

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./distilbert-text-classification",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-104770442.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


**Train the model**

In [ ]:
trainer.train()

Step,Training Loss
50,1.150500
100,0.338600
150,0.138300
200,0.078500
250,0.060000
300,0.034300


TrainOutput(global_step=336, training_loss=0.2714271964061828, metrics={'train_runtime': 67.3991, 'train_samples_per_second': 79.23, 'train_steps_per_second': 4.985, 'total_flos': 176853438489600.0, 'train_loss': 0.2714271964061828, 'epoch': 3.0})

**Evaluate & test with your own text**

In [ ]:
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.09224127978086472, 'eval_accuracy': 0.9775280898876404, 'eval_f1_macro': 0.9766573871540734, 'eval_runtime': 1.6326, 'eval_samples_per_second': 272.579, 'eval_steps_per_second': 17.151, 'epoch': 3.0}


In [ ]:
model.to("cpu")
class_names = ["Politics","Sport","Technology","Entertainment","Business",]
import torch
def predict(text):
    inputs = tokenizer(text,return_tensors="pt",truncation=True,padding=True,max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    pred_id = outputs.logits.argmax(dim=-1).item()
    return class_names[pred_id]


print(predict("The team played football very well and won the match."))
print(predict("The government announced a new policy today."))

Sport
Politics
